# Лабораторная работа №7. Классификация

Выполните следующие задания:

1. Откройте в файл в Google Colab (используйте собственный форк репозитория).
2. Решите задачи.
3. Сохраните результат в виде файла rep.ipynb в ваш репозиторий github в директорию ./les07
4. Создайте pull request в репозиторий https://github.com/chebotarevsa/dap-2024. Название pull request должно иметь формат "<Номер лабораторной работы>  <Номер группы> <ФИО>"
5. Сдайте работу в системе "Пегас", в отчет укажите ссылку на pull request

Используя набор данных titanic.csv постройте модель предсказывающую выжил или погиб пассажир. 
1.	PassengerId – Идентификатор пассажира (уникальный номер для каждого пассажира).
2.	Survived – Выжил (0 – не выжил, 1 – выжил).
3.	Pclass – Класс пассажира (1 – первый класс, 2 – второй класс, 3 – третий класс).
4.	Name – Имя (полное имя пассажира).
5.	Sex – Пол (male – мужчина, female – женщина).
6.	Age – Возраст (числовое значение, может быть дробным).
7.	SibSp – Количество родственников на борту (братьев, сестер или супругов).
8.	Parch – Количество родителей или детей на борту.
9.	Ticket – Номер билета.
10.	Fare – Стоимость билета (в фунтах стерлингов).
11.	Cabin – Номер каюты (может быть пропущен, если данные отсутствуют).
12.	Embarked – Порт посадки (C – Cherbourg, Q – Queenstown, S – Southampton).

In [ ]:
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(f"Python version: {sys.version}")
print(f"Numpy version: {np.version.version}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {mpl.__version__}")

## Извлечение данных

1. Создайте DataFarame из файла titanic.csv, в качестве индекса используйте столбец "PassengerId".

In [ ]:
import pandas as pd

file_path = './data/titanic.csv'
df = pd.read_csv(file_path, index_col='PassengerId')

print(df.head())

print("\nПропуски по столбцам:")
print(df.isnull().sum())

2. Выведете первые 7 строк набора.

In [ ]:
print(df.head(7))

### Подготовка данных

3. Выведете информацию о типах данных в наборе. Имеются ли в наборе не числовые признаки? Имеются ли в наборе данные имеющие значение null? 
    

In [ ]:
df.info()

non_numeric = df.select_dtypes(exclude=[np.number]).columns
print("\nНечисловые признаки в наборе данных:")
print(non_numeric)

missing_values = df.isnull().sum()
print("\nПропущенные значения по столбцам:")
print(missing_values[missing_values > 0])

4. Удалите не числовые признаки, которые по вашему мнению, не могут влиять на заначение выжил или погиб пассажир.

In [ ]:
df_model = df.drop(columns=['Name', 'Ticket', 'Cabin'])


print(df_model.head())

5. Вместо признака "Sex" (я надеюсь вы его не удалили 😂) Создайте два новых признака male и female которые содержат значения 0 или 1.

In [ ]:
sex_dummies = pd.get_dummies(df_model['Sex'], prefix='', prefix_sep='')

df_model = pd.concat([df_model.drop(columns=['Sex']), sex_dummies], axis=1)

print(df_model.head())

6. Удалите строки, которые содержать хотя бы одно null значение.

In [ ]:
df_model = df_model.dropna()


print(df_model.isnull().sum())
print(f"\nОставшиеся строки: {df_model.shape[0]}")

## Исследование данных

4. Нормализуйте значения признака "Fare".

In [ ]:
df_model['Fare'] = (df_model['Fare'] - df_model['Fare'].min()) / (df_model['Fare'].max() - df_model['Fare'].min())

print(df_model['Fare'].head())

5. Найдите разницу между средними значениями признака "Fare" для погибших и выживших пассажиров.

In [ ]:
fare_mean_survived = df_model[df_model['Survived'] == 1]['Fare'].mean()
fare_mean_not_survived = df_model[df_model['Survived'] == 0]['Fare'].mean()

fare_diff = fare_mean_survived - fare_mean_not_survived

print(f"Среднее Fare выживших: {fare_mean_survived:.3f}")
print(f"Среднее Fare погибших: {fare_mean_not_survived:.3f}")
print(f"Разница между средними значениями: {fare_diff:.3f}")

6. Простройте на одной оси координат гистограмы значений признака "Fare" для погибших и выживших пассажиров.

In [ ]:
import matplotlib.pyplot as plt

fare_survived = df_model[df_model['Survived'] == 1]['Fare']
fare_not_survived = df_model[df_model['Survived'] == 0]['Fare']

plt.figure(figsize=(8,6))
plt.hist(fare_survived, bins=20, alpha=0.6, label='Выжившие', color='green')
plt.hist(fare_not_survived, bins=20, alpha=0.6, label='Погибшие', color='red')
plt.xlabel('Fare (нормализованная)')
plt.ylabel('Количество пассажиров')
plt.title('Распределение стоимости билета Fare для выживших и погибших')
plt.legend()
plt.show()

7. Сформируйте набор признаков (X). Сформируйте вектор целевых значений (y).

In [ ]:
y = df_model['Survived']

X = df_model.drop(columns=['Survived'])

print(f"Форма X: {X.shape}")
print(f"Форма y: {y.shape}")

print(X.head())
print(y.head())

## Предсказательная модель

8. Разделите набор данных на два, одни для обучения модели другой для проверки. Тестовый набор должен содержать 25 процентов данных.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

9. Выполните обучение модели.

In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print(f"Точность на тренировочном наборе: {train_score:.3f}")
print(f"Точность на тестовом наборе: {test_score:.3f}")

## Проверка модели

10. Выведите мартицу ошибок

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Погиб', 'Выжил'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Матрица ошибок логистической регрессии")
plt.show()

11. Расчитайте accuracy

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy модели: {acc:.3f}")

12. Сделайте вывод о полученном результате

Вывод: модель демонстрирует базовое предсказание с неплохой точностью, но её точность ограничена отсутствием некоторых информативных признаков и возможными пропусками данных.

## Вопросы для защиты

1. Сформулируйте задачу классификации.
2. Перечислите типы классификации.
3. В чем особенность несбалансированной классификации?
4. В чем особенность мультиклассовой классификации?
5. В чем особенность бинарной классификации?
6. В чем особенность классификации по нескольким меткам?
7. Чем стратегия "Один против всех" отличается от стратегии "Один против одного"?
8. Что такое матрица ошибок (несоответствий)?
9. Как рассчитывается Accuracy?
10. Объясните алгоритм классификации K-ближайших соседей.

1. Задача классификации

Классификация — это задача машинного обучения, в которой требуется отнести объект к одной из заранее определённых категорий (классов) на основе его признаков.

Пример: предсказать, выжил пассажир Титаника или нет (Survived 0/1).

2. Типы классификации

Бинарная — два класса (например, спам/не спам).

Мультиклассовая — более двух классов (например, распознавание видов ирисов).

Многометочная (multi-label) — объект может принадлежать сразу к нескольким классам (например, теги к статье).

Иерархическая — классы имеют структуру дерева.

3. Особенность несбалансированной классификации

Классы представлены неравномерно (например, 90% одного класса, 10% другого).

Стандартные метрики (accuracy) могут быть вводящими в заблуждение, поэтому используют Precision, Recall, F1-score, ROC-AUC.

4. Особенность мультиклассовой классификации

Не два класса, а три и более.

Модели используют стратегии “один против всех” (One-vs-Rest) или “один против одного” (One-vs-One) для бинарных классификаторов.

5. Особенность бинарной классификации

Два возможных исхода (например, 0/1, да/нет).

Метрики: Accuracy, Precision, Recall, F1-score, ROC-AUC.

6. Особенность классификации по нескольким меткам

Каждый объект может одновременно принадлежать к нескольким классам.

Пример: статья может быть одновременно про спорт и политику.

Метрики: Hamming Loss, subset accuracy.

7. Разница между стратегиями

Один против всех (One-vs-Rest, OvR) — для каждого класса строится бинарный классификатор против всех остальных классов.

Один против одного (One-vs-One, OvO) — строится бинарный классификатор для каждой пары классов.

OvR быстрее при большом числе классов, OvO точнее при схожих классах.

8. Матрица ошибок (confusion matrix)

Таблица, показывающая, сколько объектов каждого класса было правильно или неправильно классифицировано.

Строки — реальные классы, столбцы — предсказанные классы.

9. Расчёт Accuracy

Пример: 80 правильно предсказанных из 100 → Accuracy = 0.8 (80%).

10. Алгоритм K-ближайших соседей (KNN)

Для нового объекта вычисляются расстояния до всех объектов обучающего набора (например, евклидово).

Выбираются K ближайших соседей.

Класс нового объекта определяется большинством классов среди этих K соседей.

Преимущества: простота, не требует обучения модели.

Недостатки: чувствителен к масштабу признаков, медленно работает на больших наборах.